# Pronunza: demo de TTS Galego con ONNX (Modelo Celtia)

Este caderno de Colab demostra como usar un modelo VITS pre-adestrado
en formato ONNX para xerar voz en galego a partir de texto.
Utiliza o modelo Jarbas/proxectonos-celtia-vits-graphemes-onnx
dispoñible en Hugging Face.

https://huggingface.co/Jarbas/proxectonos-celtia-vits-graphemes-onnx

Versión publicada e compartida en GitHub. https://github.com/gas/pronunza-tts-galego-onnx-colab

Baseado no traballo previo e discusións con Gemini.

In [ ]:
# -*- coding: utf-8 -*-

#=======================================================================
# CELDA 0: IMPORTACIÓNS INICIAIS E VERIFICACIÓNS
#=======================================================================
#@title 0. Importacións Iniciais e Verificacións

# Importacións estándar de Python
import os
import sys
import time
import json
import re
import subprocess
import tempfile
import datetime
import getpass
import string
from typing import Callable, List, Optional

# Importacións de librarías externas (instalaranse despois)
# Envolver en try/except para que esta cela non falle se se executa primeiro
try:
    import numpy as np
    import onnxruntime as ort
    import scipy
    import scipy.io.wavfile
    from huggingface_hub import login, HfFolder
    import requests
    from google.colab import drive
    from IPython.display import Audio, display
except ImportError:
    print("INFO: Algunhas librarías externas aínda non están instaladas. Instalaranse na Cela 2.")

print(f"Python version: {sys.version}")
# Comprobar se estamos en Colab (útil para certas lóxicas)
IN_COLAB = 'google.colab' in sys.modules
print(f"Executando en Google Colab: {IN_COLAB}")

Python version: 3.11.12 (main, Apr  9 2025, 08:55:54) [GCC 11.4.0]
Executando en Google Colab: True


In [ ]:
#=======================================================================
# CELDA 1: CONFIGURACIÓN DE DIRECTORIOS E MONTAXE DE GDRIVE (OPCIONAL)
#=======================================================================
#@title 1. Configuración de Directorios e Montaxe de Google Drive (Opcional) { display-mode: "form" }

# --- Parámetros de Configuración ---
#@markdown Decide se queres usar Google Drive para gardar/cargar modelos e audios:
USE_GDRIVE = False #@param {type:"boolean"}
#@markdown Ruta base do proxecto se usas Google Drive (o basepath é /content/drive/MyDrive):
GDRIVE_PROJECT_PATH = '/content/drive/MyDrive/AI/Pronunza' #@param {type:"string"}

# --- Lóxica ---
PROJECT_BASE_DIR = "/content" # Directorio por defecto en Colab

if USE_GDRIVE:
    if IN_COLAB: # Só intentar montar se estamos en Colab
        print("Intentando montar Google Drive...")
        try:
            drive.mount('/content/drive', force_remount=True)
            PROJECT_BASE_DIR = GDRIVE_PROJECT_PATH
            if not os.path.isdir(PROJECT_BASE_DIR):
                 print(f"AVISO: A ruta especificada en Drive non existe ({PROJECT_BASE_DIR}). Creando...")
            os.makedirs(PROJECT_BASE_DIR, exist_ok=True) # Crear se non existe
            print(f"✅ Usando directorio base en Google Drive: {PROJECT_BASE_DIR}")
        except Exception as e:
            print(f"⚠️ Erro ao montar ou acceder a Google Drive: {e}.")
            print("Usarase o directorio local /content en Colab.")
            PROJECT_BASE_DIR = "/content"
            os.makedirs(PROJECT_BASE_DIR, exist_ok=True)
    else:
        print("AVISO: A opción USE_GDRIVE está activa pero non se está executando en Colab. Usando directorio actual.")
        PROJECT_BASE_DIR = "." # Usar directorio actual se non estamos en Colab
else:
    print(f"ℹ️ Usando directorio local por defecto: {PROJECT_BASE_DIR}")
    os.makedirs(PROJECT_BASE_DIR, exist_ok=True)

# Cambiamos ao directorio de traballo elixido (importante para rutas relativas)
try:
    os.chdir(PROJECT_BASE_DIR)
    print(f"\nDirectorio de traballo actual:")
    !pwd # Mostra o directorio actual da shell
except Exception as e:
    print(f"⚠️ Erro ao cambiar ao directorio {PROJECT_BASE_DIR}: {e}")
    print("Permanecendo no directorio anterior.")
    PROJECT_BASE_DIR = os.getcwd() # Manter o directorio actual se falla o cambio
    print("Directorio actual:")
    !pwd

# Definimos as carpetas para modelos e audios dentro do directorio base
MODELS_DIR = os.path.join(PROJECT_BASE_DIR, "models")
WAVS_DIR = os.path.join(PROJECT_BASE_DIR, "wavs")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(WAVS_DIR, exist_ok=True)
print(f"\nCarpeta para modelos: {MODELS_DIR}")
print(f"Carpeta para audios WAV: {WAVS_DIR}")


ℹ️ Usando directorio local por defecto: /content

Directorio de traballo actual:
/content

Carpeta para modelos: /content/models
Carpeta para audios WAV: /content/wavs


In [ ]:
#=======================================================================
# CELDA 2: INSTALACIÓN DE DEPENDENCIAS
#=======================================================================
#@title 2. Instalación de Dependencias
print("Instalando librarías necesarias...")
# -q para modo silencioso
# Fixar versión de numpy pode axudar coa compatibilidade con onnxruntime ou scipy
# Proba sen fixar primeiro, se hai erros de numpy, descomenta a seguinte liña
# !pip install numpy==1.24.4 -q
!pip install onnxruntime scipy huggingface_hub requests -q
print("✅ Dependencias instaladas: onnxruntime, numpy, scipy, huggingface_hub, requests")

# Importar de novo ou por primeira vez despois da instalación
try:
    import numpy as np
    import onnxruntime as ort
    import scipy
    import scipy.io.wavfile
    from huggingface_hub import login, HfFolder
    import requests
    from IPython.display import Audio, display
    # getpass xa debería estar importado da cela 0
except ImportError as e:
     print(f"❌ ERRO: Non se puido importar unha librería despois da instalación: {e}")
     print("Pode que necesites reiniciar o entorno de execución (Runtime -> Restart session).")

print("✅ Librarías importadas correctamente.")

Instalando librarías necesarias...
✅ Dependencias instaladas: onnxruntime, numpy, scipy, huggingface_hub, requests
✅ Librarías importadas correctamente.


In [ ]:
#=======================================================================
# CELDA 3: AUTENTICACIÓN EN HUGGING FACE
#=======================================================================
#@title 3. Autenticación en Hugging Face
print("Necesitas un token de Hugging Face para descargar o modelo.")
print("Podes xerar un na túa conta: https://huggingface.co/settings/tokens (permiso 'read')")

# Comprobar se xa hai un token gardado
hf_token = HfFolder.get_token()
if hf_token:
    print("ℹ️ Xa existe un token de Hugging Face gardado.")
else:
    print("\nPega o teu token de Hugging Face abaixo e presiona Enter:")
    hf_token_input = getpass.getpass()
    if hf_token_input:
        try:
            # add_to_git_credential=False evita un aviso en Colab
            login(token=hf_token_input, add_to_git_credential=False)
            print("✅ Token gardado para esta sesión.")
        except Exception as e:
            print(f"⚠️ Erro ao gardar o token: {e}")
    else:
        print("⚠️ Non se introduciu token. A descarga do modelo pode fallar se require autenticación.")

# Verifica manualmente que aceptaches os termos na páxina do modelo:
# https://huggingface.co/Jarbas/proxectonos-celtia-vits-graphemes-onnx
print("\n⚠️ Asegúrate de ter aceptado os termos na páxina do modelo en Hugging Face.")


Necesitas un token de Hugging Face para descargar o modelo.
Podes xerar un na túa conta: https://huggingface.co/settings/tokens (permiso 'read')
ℹ️ Xa existe un token de Hugging Face gardado.

⚠️ Asegúrate de ter aceptado os termos na páxina do modelo en Hugging Face.


In [ ]:
#=======================================================================
# CELDA 4: DESCARGA DO MODELO ONNX E CONFIGURACIÓN
#=======================================================================
#@title 4. Descarga do Modelo ONNX e Configuración

# Rutas completas onde gardar os arquivos (usando MODELS_DIR definido na Celda 1)
config_file_path = os.path.join(MODELS_DIR, "config.json")
model_file_path = os.path.join(MODELS_DIR, "model.onnx")

# URL dos arquivos no repositorio de Jarbas
config_url = "https://huggingface.co/Jarbas/proxectonos-celtia-vits-graphemes-onnx/resolve/main/config.json"
model_url = "https://huggingface.co/Jarbas/proxectonos-celtia-vits-graphemes-onnx/resolve/main/model.onnx"

def download_file(url, dest_path):
    """Descarga un arquivo desde unha URL a unha ruta destino."""
    try:
        filename = os.path.basename(dest_path)
        print(f"Descargando {filename}...")
        headers = {}
        token = HfFolder.get_token()
        if token:
            headers["Authorization"] = f"Bearer {token}"

        # Realizar a petición GET
        response = requests.get(url, headers=headers, stream=True, timeout=60) # Timeout aumentado
        response.raise_for_status() # Lanza erro se a descarga falla (4xx, 5xx)

        # Escribir o contido ao arquivo
        with open(dest_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"✅ {filename} descargado con éxito en {dest_path}")
        return True # Indica éxito

    except requests.exceptions.RequestException as e:
        print(f"❌ Erro ao descargar {url}: {e}")
        # Se é un erro 401 ou 403, pode ser problema de token ou permisos
        if response.status_code in [401, 403]:
             print("   Verifica o teu token de Hugging Face e que aceptaches os termos do modelo.")
    except IOError as e:
        print(f"❌ Erro ao gardar en {dest_path}: {e}")
    except Exception as e:
        print(f"❌ Erro inesperado durante a descarga de {os.path.basename(dest_path)}: {e}")
    return False # Indica fallo

# Descargamos os arquivos SÓ se non existen xa
print("\nComprobando/Descargando arquivos do modelo...")
config_ok = os.path.exists(config_file_path)
model_ok = os.path.exists(model_file_path)

if not config_ok:
    config_ok = download_file(config_url, config_file_path)
else:
    print(f"ℹ️ {os.path.basename(config_file_path)} xa existe.")

if not model_ok:
    model_ok = download_file(model_url, model_file_path)
else:
    print(f"ℹ️ {os.path.basename(model_file_path)} xa existe.")

# Verificación final
if config_ok and model_ok:
    print("\n✅ Arquivos necesarios listos en:")
    !ls -lh "{MODELS_DIR}"
else:
    print("\n❌ Faltan arquivos esenciais do modelo. A inicialización posterior pode fallar.")



Comprobando/Descargando arquivos do modelo...
Descargando config.json...
✅ config.json descargado con éxito en /content/models/config.json
Descargando model.onnx...
✅ model.onnx descargado con éxito en /content/models/model.onnx

✅ Arquivos necesarios listos en:
total 125M
-rw-r--r-- 1 root root 7.3K May  1 13:19 config.json
-rw-r--r-- 1 root root 125M May  1 13:19 model.onnx


In [ ]:
#=======================================================================
# CELDA 5: DEFINICIÓNS DAS CLASES PYTHON PARA INFERENCIA ONNX
#=======================================================================
#@title 5 Definicións das Clases Python para Inferencia ONNX# Regular expression matching whitespace:

print("Definindo clases Graphemes, TTSTokenizer, VitsOnnxInference...")

_whitespace_re = re.compile(r"\s+")


class Graphemes:
    """Xestiona o vocabulario e mapeo para grafemas."""
    def __init__(
            self,
            characters: str = None,
            punctuations: str = None,
            pad: str = None,
            eos: str = None,
            bos: str = None,
            blank: str = "<BLNK>",
            is_unique: bool = False,
            is_sorted: bool = True,
    ) -> None:
        self._characters = characters
        self._punctuations = punctuations
        self._pad = pad
        self._eos = eos
        self._bos = bos
        self._blank = blank
        self.is_unique = is_unique
        self.is_sorted = is_sorted
        self._create_vocab()

    @property
    def pad_id(self) -> int:
        return self.char_to_id(self.pad) if self.pad else len(self.vocab)

    @property
    def blank_id(self) -> int:
        return self.char_to_id(self.blank) if self.blank else len(self.vocab)

    @property
    def eos_id(self) -> int:
        return self.char_to_id(self.eos) if self.eos else len(self.vocab)

    @property
    def bos_id(self) -> int:
        return self.char_to_id(self.bos) if self.bos else len(self.vocab)

    @property
    def characters(self):
        return self._characters

    @characters.setter
    def characters(self, characters):
        self._characters = characters
        self._create_vocab()

    @property
    def punctuations(self):
        return self._punctuations

    @punctuations.setter
    def punctuations(self, punctuations):
        self._punctuations = punctuations
        self._create_vocab()

    @property
    def pad(self):
        return self._pad

    @pad.setter
    def pad(self, pad):
        self._pad = pad
        self._create_vocab()

    @property
    def eos(self):
        return self._eos

    @eos.setter
    def eos(self, eos):
        self._eos = eos
        self._create_vocab()

    @property
    def bos(self):
        return self._bos

    @bos.setter
    def bos(self, bos):
        self._bos = bos
        self._create_vocab()

    @property
    def blank(self):
        return self._blank

    @blank.setter
    def blank(self, blank):
        self._blank = blank
        self._create_vocab()

    @property
    def vocab(self):
        return self._vocab

    @vocab.setter
    def vocab(self, vocab):
        self._vocab = vocab
        self._char_to_id = {char: idx for idx, char in enumerate(self.vocab)}
        self._id_to_char = {
            idx: char for idx, char in enumerate(self.vocab)  # pylint: disable=unnecessary-comprehension
        }

    @property
    def num_chars(self):
        return len(self._vocab)

    def _create_vocab(self):
        self._vocab = [self._pad] + list(self._punctuations) + list(self._characters) + [self._blank]
        self._char_to_id = {char: idx for idx, char in enumerate(self.vocab)}
        # pylint: disable=unnecessary-comprehension
        self._id_to_char = {idx: char for idx, char in enumerate(self.vocab)}

    def char_to_id(self, char: str) -> int:
        try:
            return self._char_to_id[char]
        except KeyError as e:
            raise KeyError(f" [!] {repr(char)} is not in the vocabulary.") from e

    def id_to_char(self, idx: int) -> str:
        return self._id_to_char[idx]


class TTSTokenizer:
    """Tokenizador para TTS: Normaliza texto e convirte a IDs."""
    """🐸TTS tokenizer to convert input characters to token IDs and back.
    Token IDs for OOV chars are discarded but those are stored in `self.not_found_characters` for later.
    Args:
        characters (Characters):
            A Characters object to use for character-to-ID and ID-to-character mappings.
        text_cleaner (callable):
            A function to pre-process the text before tokenization and phonemization. Defaults to None.
    """

    def __init__(
            self,
            text_cleaner: Callable = None,
            characters: Graphemes = None,
            add_blank: bool = False,
            use_eos_bos=False,
    ):
        self.text_cleaner = text_cleaner
        self.add_blank = add_blank
        self.use_eos_bos = use_eos_bos
        self.characters = characters
        self.not_found_characters = []

    @property
    def characters(self):
        return self._characters

    @characters.setter
    def characters(self, new_characters):
        self._characters = new_characters
        self.pad_id = self.characters.char_to_id(self.characters.pad) if self.characters.pad else None
        self.blank_id = self.characters.char_to_id(self.characters.blank) if self.characters.blank else None

    def encode(self, text: str) -> List[int]:
        """Encodes a string of text as a sequence of IDs."""
        token_ids = []
        for char in text:
            try:
                idx = self.characters.char_to_id(char)
                token_ids.append(idx)
            except KeyError:
                # discard but store not found characters
                if char not in self.not_found_characters:
                    self.not_found_characters.append(char)
                    print(text)
                    print(f" [!] Character {repr(char)} not found in the vocabulary. Discarding it.")
        return token_ids

    def text_to_ids(self, text: str) -> List[int]:  # pylint: disable=unused-argument
        """Converts a string of text to a sequence of token IDs.

        Args:
            text(str):
                The text to convert to token IDs.

        1. Text normalization
        3. Add blank char between characters
        4. Add BOS and EOS characters
        5. Text to token IDs
        """
        if self.text_cleaner is not None:
            text = self.text_cleaner(text)
        text = self.encode(text)
        if self.add_blank:
            text = self.intersperse_blank_char(text, True)
        if self.use_eos_bos:
            text = self.pad_with_bos_eos(text)
        return text

    def pad_with_bos_eos(self, char_sequence: List[str]):
        """Pads a sequence with the special BOS and EOS characters."""
        return [self.characters.bos_id] + list(char_sequence) + [self.characters.eos_id]

    def intersperse_blank_char(self, char_sequence: List[str], use_blank_char: bool = False):
        """Intersperses the blank character between characters in a sequence.

        Use the ```blank``` character if defined else use the ```pad``` character.
        """
        char_to_use = self.characters.blank_id if use_blank_char else self.characters.pad
        result = [char_to_use] * (len(char_sequence) * 2 + 1)
        result[1::2] = char_sequence
        return result


class VitsOnnxInference:
    """Clase para cargar un modelo VITS ONNX e realizar inferencia TTS."""
    def __init__(self, onnx_model_path: str, config_path: str, cuda=False):
        self.config = {}
        if config_path:
            with open(config_path) as f:
                self.config = json.load(f)
        providers = [
            "CPUExecutionProvider"
            if cuda is False
            else ("CUDAExecutionProvider", {"cudnn_conv_algo_search": "DEFAULT"})
        ]
        sess_options = ort.SessionOptions()
        self.onnx_sess = ort.InferenceSession(
            onnx_model_path,
            sess_options=sess_options,
            providers=providers,
        )

        _pad = self.config.get("characters", {}).get("pad", "_")
        _punctuations = self.config.get("characters", {}).get("punctuations", "!\"(),-.:;?\u00a1\u00bf ")
        _letters = self.config.get("characters", {}).get("characters",
                                                         "ABCDEFGHIJKLMNOPQRSTUVXYZabcdefghijklmnopqrstuvwxyz\u00c1\u00c9\u00cd\u00d3\u00da\u00e1\u00e9\u00ed\u00f1\u00f3\u00fa\u00fc")

        vocab = Graphemes(characters=_letters,
                          punctuations=_punctuations,
                          pad=_pad)

        self.tokenizer = TTSTokenizer(
            text_cleaner=self.normalize_text,
            characters=vocab,
            add_blank=self.config.get("add_blank", True),
            use_eos_bos=False,
        )

    @staticmethod
    def normalize_text(text: str) -> str:
        """Basic pipeline that lowercases and collapses whitespace without transliteration."""
        text = text.lower()
        text = text.replace(";", ",")
        text = text.replace("-", " ")
        text = text.replace(":", ",")
        text = re.sub(r"[\<\>\(\)\[\]\"]+", "", text)
        text = re.sub(_whitespace_re, " ", text).strip()
        return text

    def inference_onnx(self, text: str):
        """ONNX inference"""
        x = np.asarray(
            self.tokenizer.text_to_ids(text),
            dtype=np.int64,
        )[None, :]

        x_lengths = np.array([x.shape[1]], dtype=np.int64)

        scales = np.array(
            [self.config.get("inference_noise_scale", 0.667),
             self.config.get("length_scale", 1.0),
             self.config.get("inference_noise_scale_dp", 1.0), ],
            dtype=np.float32,
        )
        input_params = {"input": x, "input_lengths": x_lengths, "scales": scales}

        audio = self.onnx_sess.run(
            ["output"],
            input_params,
        )
        return audio[0][0]

    @staticmethod
    def save_wav(wav: np.ndarray, path: str, sample_rate: int = 16000) -> None:
        """Save float waveform to a file using Scipy.

        Args:
            wav (np.ndarray): Waveform with float values in range [-1, 1] to save.
            path (str): Path to a output file.
        """
        wav_norm = wav * (32767 / max(0.01, np.max(np.abs(wav))))
        wav_norm = wav_norm.astype(np.int16)
        scipy.io.wavfile.write(path, sample_rate, wav_norm)

    def synth(self, text: str, path: str):
        wavs = self.inference_onnx(text)
        self.save_wav(wavs[0], path, self.config.get("sample_rate", 16000))

print("✅ Definicións de clase para inferencia ONNX cargadas.")


Definindo clases Graphemes, TTSTokenizer, VitsOnnxInference...
✅ Definicións de clase para inferencia ONNX cargadas.


In [ ]:
#=======================================================================
# CELDA 6: INICIALIZACIÓN DO MOTOR TTS (CARGA DO MODELO)
#=======================================================================
#@title 6. Inicialización do Motor TTS (Carga do Modelo)

print("--- Inicializando o Motor TTS ONNX (Carga do Modelo) ---")

# Usamos as variables coas rutas definidas na cela 4 ("Descarga do Modelo")
# Asegurámonos de que están definidas e os arquivos existen
onnx_tts_engine = None # Inicializar a None por si falla a carga
load_error = False     # Flag para saber se houbo erro

# Comprobar se as variables de ruta están definidas (deberían estar se a cela 4 executouse)
if 'model_file_path' not in locals() or 'config_file_path' not in locals():
     print("❌ ERRO: As variables model_file_path ou config_file_path non están definidas.")
     print("Asegúrate de executar a cela 4 (Descarga do Modelo) primeiro.")
     load_error = True
# Comprobar se os arquivos realmente existen
elif not os.path.exists(model_file_path):
    print(f"❌ ERRO Fatal: Non se atopou o modelo ONNX en {model_file_path}.")
    print("Verifica se a descarga na cela 4 foi exitosa.")
    load_error = True
elif not os.path.exists(config_file_path):
    print(f"❌ ERRO Fatal: Non se atopou o config.json en {config_file_path}.")
    print("Verifica se a descarga na cela 4 foi exitosa.")
    load_error = True
else:
    # Se as rutas están ben e os arquivos existen, intentar cargar
    try:
        # Comprobar se a clase VitsOnnxInference está definida
        if 'VitsOnnxInference' in locals() or 'VitsOnnxInference' in globals():
            # Creamos a instancia global que usaremos despois
            # Usamos cuda=True para intentar usar a GPU de Colab
            onnx_tts_engine = VitsOnnxInference(onnx_model_path=model_file_path,
                                                config_path=config_file_path,
                                                cuda=True)
            # A mensaxe de éxito imprímese ao final se non hai erros
        else:
             load_error = True
             print("❌ ERRO: A clase VitsOnnxInference non parece estar definida. Executa a cela 5 coas definicións.")

    except Exception as e:
        load_error = True
        print(f"--- ❌ ERRO FATAL durante a inicialización do motor TTS ONNX ---")
        print(f"Erro: {e}")
        import traceback
        traceback.print_exc() # Imprimir traceback completo para máis detalles
        print("----------------------------------------------------------")

# Mensaxe final sobre o estado da inicialización
if not load_error and onnx_tts_engine:
     print("\n✅ Motor TTS ONNX inicializado e listo.")
elif not load_error and not onnx_tts_engine:
     print("❓ A inicialización rematou sen erros pero onnx_tts_engine non foi creado. Revisa a lóxica da cela 5.")
     load_error = True # Marcar erro se non se creou
else:
     print("\n⚠️ O motor TTS non puido inicializarse. A síntese non funcionará.")



--- Inicializando o Motor TTS ONNX (Carga do Modelo) ---

✅ Motor TTS ONNX inicializado e listo.


In [ ]:
#=======================================================================
# CELDA 7: EXECUCIÓN DO MODELO TTS
#=======================================================================
#@title 7. Execución do Modelo TTS { display-mode: "form" }

# --- CONFIGURACIÓN DA SÍNTESE ---
#@markdown Introduce o texto en galego que queres sintetizar:
texto_para_sintetizar = "Ola, benvidos á proba de Pronunza! e qué é iso?... Un sistema de texto a voz para galego, baseado en modelos abertos." #@param {type:"string"}
#@markdown Nome base para o arquivo de saída (engadirase timestamp):
nome_base_arquivo = "pronunza_output" #@param {type:"string"}

# --- EXECUCIÓN ---
# Comproba se o motor cargou ben na cela anterior
# Usamos 'onnx_tts_engine' directamente se está definido e non é None
tts_engine_disponible = 'onnx_tts_engine' in locals() and onnx_tts_engine is not None and not load_error

if tts_engine_disponible:
    try:
        print("\n--- Iniciando Síntese ---")

        # Validar texto de entrada
        if not texto_para_sintetizar or not texto_para_sintetizar.strip():
             print("❌ ERRO: O texto para sintetizar non pode estar vacío.")
        else:
            # Crear nome de arquivo único con timestamp
            now = datetime.datetime.now().strftime("%Y%m%d_%H%M%S_%f") # Engade microsegundos
            # Limpar nome base por se acaso
            safe_base_name = "".join(c for c in nome_base_arquivo if c.isalnum() or c in ('_', '-')).rstrip()
            safe_base_name = safe_base_name if safe_base_name else "output"
            # Usar WAVS_DIR definido na cela 1
            archivo_salida = os.path.join(WAVS_DIR, f"{safe_base_name}_{now}.wav")

            print(f"Texto a sintetizar: '{texto_para_sintetizar}'")
            print(f"Gardarase como: {archivo_salida}")

            # Medir tempo de síntese
            start_synth_time = time.time()

            # Chamar ao método synth que fai todo
            # Pasamos o texto e a ruta de saída
            onnx_tts_engine.synth(text=texto_para_sintetizar, path=archivo_salida)

            end_synth_time = time.time()

            # Comprobar se o arquivo foi creado e ten contido antes de dicir que foi exitoso
            if os.path.exists(archivo_salida) and os.path.getsize(archivo_salida) > 44: # 44 bytes é o tamaño mínimo dun header WAV
                 print(f"\n✅ Síntese completada en {end_synth_time - start_synth_time:.2f} segundos.")
                 # Mostrar reprodutor en Colab
                 print("\n▶️ Reproducindo audio xerado:")
                 display(Audio(archivo_salida, autoplay=False))
            else:
                 print(f"\n⚠️ AVISO: A síntese rematou pero o arquivo de audio ({archivo_salida}) non se creou correctamente ou está vacío.")
            print("----------------------------------------------------------")

    except Exception as e:
        print(f"\n--- ❌ ERRO DURANTE A SÍNTESE ---")
        print(f"Erro: {e}")
        import traceback
        traceback.print_exc() # Imprime o traceback completo
        print("------------------------------------")
else:
    print("\n❌ ERRO: O motor TTS ONNX non foi inicializado correctamente.")
    print("Revisa a saída da cela '6. Inicialización do Motor TTS' para ver os erros.")
    print("Asegúrate de executar todas as celas anteriores en orde.")



--- Iniciando Síntese ---
Texto a sintetizar: 'Ola, benvidos á proba de Pronunza! e qué é iso?... Un sistema de texto a voz para galego, baseado en modelos abertos.'
Gardarase como: /content/wavs/pronunza_output_20250501_133352_279659.wav

✅ Síntese completada en 9.07 segundos.

▶️ Reproducindo audio xerado:


----------------------------------------------------------


**Conclusións e Notas de Versión**

Este caderno demostra un fluxo de traballo completo para usar un modelo TTS ONNX
pre-adestrado para xerar voz en galego.

**Cambios Principais nesta Versión (v0.8):**

1.  **`import os` Movido a Cela 0:** Resolve definitivamente o `NameError`.
2.  **Importacións Agrupadas:** A Cela 0 importa librarías estándar. A Cela 2 instala as externas e as importa de novo para asegurar disponibilidade.
3.  **Cela 6 Esplícita para Inicialización:** Créase a instancia `onnx_tts_engine` na sua propia cela, despóis de definir as clases.
4.  **Comprobacións Robustas:** Engádense máis `try...except` e comprobaciós (si existen os arquivos, si o motor se cargou) para dar mensaxes de erro máis claros.
5.  **Descarga Mellorada (Cela 4):** Úsase `requests` e o token de HF para descargas potencialmente autenticadas.
6.  **Nomes de Arquivo WAV (Cela 7):** Úsase timestamp con microsegundos para asegurar nomes únicos.
7.  **Títulos e Comentarios:** Úsase `@title` para a estrutura visual en Colab e tamén se engadiron/melloraron comentarios.